# Notebook 10 · Đo tỷ lệ hàm bị cắt ở giới hạn 768 token (khoảng 2–5 phút, không cần GPU)

Mục đích: báo cáo đang ghi "chưa kiểm tra trực tiếp tỷ lệ hàm bị cắt ở giới hạn 768 token". Notebook này mã hóa **toàn bộ 18.694 hàm** của tập sạch bằng **đúng pipeline đã dùng khi huấn luyện** (`UniXcoderInputPipeline` của repo: `<s> <encoder-only> </s> CODE </s>`, tối đa 764 token mã + 4 token đặc biệt = 768, cắt bên phải, bộ tách từ của `microsoft/unixcoder-base-nine` ở revision `5f0dc256…`) rồi đếm số hàm bị cắt.

Ghi chú: file `configs/tokenization_v1.json` của repo đã ghi số đo ở giai đoạn đầu trên **train + validation (18.139 dòng)**: chuỗi dài nhất 720 token, 0% vượt 768. Tập test chưa được đo khi đó. Notebook này đo cả ba phần trên tập sạch.

Chuẩn bị:
1. Add Input → dataset `mvd-grouped-cv-inputs` (cùng dataset của các notebook 01–03).
2. Settings: Accelerator = **None** (không cần GPU, không tốn quota GPU), **Internet = On** (để tải bộ tách từ).
3. Bấm **Run All**. Kết quả in ở cell cuối và nằm trong file `token_length_results.zip` ở tab Output.

## 1. Môi trường

In [ ]:
!pip install -q transformers pandas
import transformers, pandas
print('transformers', transformers.__version__, '| pandas', pandas.__version__)

## 2. Mã nguồn và dữ liệu thô (giống notebook 01)

In [ ]:
import glob, os, shutil, zipfile
from pathlib import Path
REPO = Path('/kaggle/working/repo')
if not (REPO / 'src/preprocessing/dataset_loader.py').exists():
    hits = glob.glob('/kaggle/input/**/src/preprocessing/dataset_loader.py', recursive=True)
    if hits:
        shutil.copytree(Path(hits[0]).parents[2], REPO)
    else:
        zips = glob.glob('/kaggle/input/**/multilingual-vulnerability-detection*.zip', recursive=True)
        assert zips, 'Không thấy repo trong /kaggle/input'
        tmp = Path('/kaggle/working/_repo_tmp'); tmp.mkdir(exist_ok=True)
        with zipfile.ZipFile(zips[0]) as z: z.extractall(tmp)
        inner = [p for p in tmp.iterdir() if p.is_dir()]
        shutil.move(str(inner[0] if len(inner) == 1 else tmp), str(REPO))
raw_dir = REPO / 'data/raw'; raw_dir.mkdir(parents=True, exist_ok=True)
NEED = ['train_w_line.csv', 'val_w_line.csv', 'test_w_line.csv']
if not all((raw_dir / n).exists() for n in NEED):
    # Kaggle có thể đã tự giải nén data_raw_for_colab.zip thành thư mục: tìm 3 file CSV ở bất kỳ đâu
    found = {}
    for n in NEED:
        cands = [Path(p) for p in glob.glob(f'/kaggle/input/**/{n}', recursive=True)]
        cands = [p for p in cands if p.stat().st_size > 1_000_000]
        if cands: found[n] = cands[0]
    if len(found) == len(NEED):
        for n, src in found.items(): shutil.copy2(src, raw_dir / n)
    else:  # trường hợp còn nguyên file zip
        zp = list(REPO.rglob('data_raw_for_colab.zip')) + [Path(p) for p in glob.glob('/kaggle/input/**/data_raw_for_colab.zip', recursive=True)]
        assert zp, 'Không thấy 3 file *_w_line.csv cũng không thấy data_raw_for_colab.zip trong /kaggle/input'
        with zipfile.ZipFile(zp[0]) as z: z.extractall(raw_dir)
import pandas as pd
rows = sum(len(pd.read_csv(raw_dir / n, usecols=[0])) for n in NEED)
print('Dữ liệu thô:', {n: (raw_dir / n).stat().st_size // 1_000_000 for n in NEED}, 'MB | tổng dòng:', rows)
assert rows == 20165, f'Số dòng {rows} != 20165: sai file dữ liệu thô'
FOLDS = glob.glob('/kaggle/input/**/groups_and_folds_v1.csv', recursive=True)
FOLDS = FOLDS[0] if FOLDS else None
print('File fold:', FOLDS)
os.chdir(REPO)

## 3. Mã hóa toàn bộ tập sạch bằng pipeline huấn luyện

In [ ]:
import sys, time, json
import numpy as np
sys.path.insert(0, str(REPO))
from src.preprocessing.dataset_loader import MVDDatasetLoader
from src.preprocessing.tokenization import UniXcoderInputPipeline

clean = MVDDatasetLoader().clean_exact_hash(copy=True)
print('Số hàm sạch:', len(clean))
assert len(clean) == 18694, 'Sai tập sạch'
if FOLDS:
    f = pd.read_csv(FOLDS, usecols=['unit_id'])
    assert set(f.unit_id) == set(clean.unit_id), 'unit_id không khớp groups_and_folds_v1.csv'
    print('unit_id khớp groups_and_folds_v1.csv')

pipe = UniXcoderInputPipeline()
print('Mô hình:', pipe.model_name, '| revision:', pipe.model_revision,
      '| max_length:', pipe.max_length, '| ngân sách token mã:', pipe.code_token_budget)

t0 = time.time(); rows = []
for uid, code, lang, split in zip(clean.unit_id, clean.func, clean.language, clean.split):
    e = pipe.encode_function(code)
    rows.append((uid, lang, split, e.code_token_length_untruncated,
                 e.code_token_length_untruncated + pipe.special_token_overhead,
                 e.sequence_length, e.truncated))
tok = pd.DataFrame(rows, columns=['unit_id', 'language', 'split', 'code_tokens',
                                  'full_length_untruncated', 'length_used', 'truncated'])
print(f'Mã hóa xong trong {time.time() - t0:.0f} giây')

## 4. Kết quả

In [ ]:
def summary(g):
    L = g.full_length_untruncated
    return pd.Series({'số hàm': len(g), 'trung vị': int(L.median()), 'P90': int(L.quantile(.9)),
                      'P99': int(L.quantile(.99)), 'dài nhất': int(L.max()),
                      '% > 512': round(100 * (L > 512).mean(), 2),
                      'số hàm bị cắt (> 768)': int(g.truncated.sum()),
                      '% bị cắt': round(100 * g.truncated.mean(), 3)})
by_lang = tok.groupby('language').apply(summary)
by_lang.loc['TỔNG'] = summary(tok)
by_split = tok.groupby('split').apply(summary)
INT = ['số hàm', 'trung vị', 'P90', 'P99', 'dài nhất', 'số hàm bị cắt (> 768)']
by_lang[INT] = by_lang[INT].astype(int); by_split[INT] = by_split[INT].astype(int)
pd.set_option('display.width', 200)
print('=== Theo ngôn ngữ (độ dài tính cả 4 token đặc biệt) ==='); print(by_lang.to_string())
print('\n=== Theo phần dữ liệu gốc ==='); print(by_split.to_string())
n_cut = int(tok.truncated.sum())
print(f'\nKẾT LUẬN: {n_cut}/{len(tok)} hàm ({100 * n_cut / len(tok):.3f}%) bị cắt ở giới hạn 768 token; '
      f'chuỗi dài nhất (chưa cắt) là {int(tok.full_length_untruncated.max())} token.')
if n_cut:
    print(tok[tok.truncated].sort_values('full_length_untruncated', ascending=False).head(20).to_string(index=False))

out = Path('/kaggle/working/token_length_results'); out.mkdir(exist_ok=True)
tok.to_csv(out / 'token_lengths.csv', index=False)
by_lang.to_csv(out / 'summary_by_language.csv'); by_split.to_csv(out / 'summary_by_split.csv')
json.dump({'n_units': len(tok), 'n_truncated': n_cut, 'max_full_length': int(tok.full_length_untruncated.max()),
           'model': pipe.model_name, 'revision': pipe.model_revision, 'transformers': transformers.__version__},
          open(out / 'summary.json', 'w'), ensure_ascii=False, indent=1)
shutil.make_archive('/kaggle/working/token_length_results', 'zip', '/kaggle/working', 'token_length_results')
from IPython.display import FileLink, display
display(FileLink('/kaggle/working/token_length_results.zip'))